# Fine-tuning DistilBERT (Multilingual)

Model: distilbert-base-multilingual-cased

## 1. Verifikimi i GPU-se

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## 2. Path-et e dataset-it (input files)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/val.csv
/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/train.csv
/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/test.csv


## 3. Instalimi dhe verifikimi i librarive

In [ ]:
import transformers
import datasets
print("transformers version:", transformers.__version__)
print("datasets version:", datasets.__version__)

transformers version: 5.0.0
datasets version: 5.0.0


## 4. Ngarkimi i dataset-it dhe pergatitja e labels

In [ ]:
import pandas as pd

DATA_PATH = '/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed'

train_df = pd.read_csv(f'{DATA_PATH}/train.csv')
val_df = pd.read_csv(f'{DATA_PATH}/val.csv')
test_df = pd.read_csv(f'{DATA_PATH}/test.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# krijo mapping label -> integer (transformer-at kërkojnë labels numerike)
kategorite = sorted(train_df['label'].unique())
label2id = {label: i for i, label in enumerate(kategorite)}
id2label = {i: label for label, i in label2id.items()}

print("\nLabel mapping:")
print(label2id)

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

train_df.head()

Train: (75480, 4)
Validation: (16174, 4)
Test: (16175, 4)

Label mapping:
{'Botë': 0, 'Ekonomi': 1, 'Kulturë': 2, 'Kuriozitete': 3, 'Lifestyle': 4, 'Politikë': 5, 'Showbiz': 6, 'Shëndet': 7, 'Sport': 8}


,text_final,label,source,date,label_id
0,Gjëra tronditëse që mësuam nga Wikileaks: Fiks...,Botë,Bota_al,2016-10-14T16:17:57,0
1,A është argjentinasja ajo që i shton gjasat që...,Showbiz,GazetaBlic,2017-01-18 11:35:55,6
2,"Ronaldo, Aguerro apo Benzema? Cana tregon se c...",Sport,GazetaBlic,2017-01-23 11:50:40,8
3,"Luftojnë tensionin e lartë të gjakut, këto jan...",Shëndet,Lajme_al,2021-04-02 17:33:17,7
4,"Huamarrja me euro, qeveria ndryshon monedhën e...",Ekonomi,Lajme_al,2016-11-21 17:23:46,1


## 5. Tokenizimi

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[['text_final', 'label_id']])
val_ds = Dataset.from_pandas(val_df[['text_final', 'label_id']])
test_ds = Dataset.from_pandas(test_df[['text_final', 'label_id']])

def tokenize_function(examples):
    return tokenizer(examples['text_final'], truncation=True, max_length=256, padding='max_length')

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.rename_column('label_id', 'labels')
val_ds = val_ds.rename_column('label_id', 'labels')
test_ds = test_ds.rename_column('label_id', 'labels')

train_ds = train_ds.remove_columns(['text_final'])
val_ds = val_ds.remove_columns(['text_final'])
test_ds = test_ds.remove_columns(['text_final'])

print(train_ds)
print("\nShembull i tokenizuar:")
print(train_ds[0])

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/75480 [00:00<?, ? examples/s]

Map:   0%|          | 0/16174 [00:00<?, ? examples/s]

Map:   0%|          | 0/16175 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 75480
})

Shembull i tokenizuar:
{'labels': 0, 'input_ids': [101, 144, 29713, 10288, 53193, 19201, 93755, 13345, 13079, 73125, 10147, 10184, 95469, 25277, 11676, 131, 36448, 12287, 10500, 10801, 69335, 118, 10323, 119, 72786, 54942, 173, 37651, 10511, 55346, 10226, 15743, 23634, 100, 11034, 194, 19223, 10526, 100, 12329, 45394, 13345, 182, 10317, 46689, 10687, 109292, 55420, 117, 151, 10761, 17424, 117, 10687, 22771, 11113, 119, 19561, 24205, 148, 13020, 54193, 10730, 77586, 90724, 10126, 117, 10148, 74208, 13345, 22265, 10730, 10146, 10240, 29713, 10687, 10398, 10381, 13345, 10149, 10511, 12240, 73099, 11498, 11227, 11546, 41931, 10885, 12240, 67192, 117, 41740, 10149, 10511, 18910, 44826, 51892, 34277, 10511, 54942, 10398, 18533, 173, 109292, 55420, 10486, 119, 16207, 45394, 74208, 117, 161, 19792, 72923, 10972, 19570, 179, 13516, 32049, 12819, 177, 12931, 10486, 28353, 11034, 10588, 15

## 6. Ngarkimi i modelit dhe konfigurimi i metrics

In [ ]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForSequenceClassification
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(kategorite),
    id2label=id2label,
    label2id=label2id
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average='macro')
    return {
        'accuracy': accuracy['accuracy'],
        'f1_macro': f1_macro['f1']
    }

print("Modeli u ngarkua. Numri i klasave:", model.config.num_labels)

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modeli u ngarkua. Numri i klasave: 9


## 7. Training Arguments dhe Trainer

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='/kaggle/working/distilbert-results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=100,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Trainer u konfigurua. Gati për fine-tuning.")

Trainer u konfigurua. Gati për fine-tuning.


## 8. Trajnimi

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.927780,0.901979,0.844874,0.844318
2,0.743208,0.822401,0.858786,0.857506
3,0.541395,0.831547,0.863237,0.861527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=7077, training_loss=0.8618791150864831, metrics={'train_runtime': 3847.2733, 'train_samples_per_second': 58.857, 'train_steps_per_second': 1.839, 'total_flos': 1.499983114696704e+16, 'train_loss': 0.8618791150864831, 'epoch': 3.0})

## 9. Vleresimi final mbi test set

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_ds)
print("Rezultatet mbi TEST SET:")
for key, value in test_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Rezultatet mbi TEST SET:
eval_loss: 0.8611657023429871
eval_accuracy: 0.858856259659969
eval_f1_macro: 0.8570067001747769
eval_runtime: 85.4568
eval_samples_per_second: 189.277
eval_steps_per_second: 2.961
epoch: 3.0


## 10. Ruajtja e modelit dhe rezultateve

In [ ]:
import json

# ruaj modelin dhe tokenizer-in
trainer.save_model('/kaggle/working/distilbert-final-model')
tokenizer.save_pretrained('/kaggle/working/distilbert-final-model')

# ruaj rezultatet për krahasim të mëvonshëm
distilbert_rezultate = {
    'model': 'distilbert-base-multilingual-cased',
    'test_accuracy': test_results['eval_accuracy'],
    'test_f1_macro': test_results['eval_f1_macro'],
    'test_loss': test_results['eval_loss'],
    'training_time_sec': 3847.27,
    'epochs': 3
}

with open('/kaggle/working/distilbert_rezultate.json', 'w') as f:
    json.dump(distilbert_rezultate, f, indent=2)

print("Modeli dhe rezultatet u ruajtën.")
print(distilbert_rezultate)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modeli dhe rezultatet u ruajtën.
{'model': 'distilbert-base-multilingual-cased', 'test_accuracy': 0.858856259659969, 'test_f1_macro': 0.8570067001747769, 'test_loss': 0.8611657023429871, 'training_time_sec': 3847.27, 'epochs': 3}
